# Build a Tool-Using Agent (No Framework)

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 12 §12.3–12.4 · type: walkthrough*  🔧 *This is the chapter's Build.*

In 12-01 you traced one tool call by hand. Now you'll **automate** it: a reusable
`while`-loop agent that registers tools, validates arguments, routes the model's
requests to executors, loops until the model is done, and **stops safely**. Forty
lines of logic, and it's a real agent — the engine the capstone grows around.

## 🧠 Why this matters

Every agent framework you'll meet later — LangGraph, Pydantic AI, the SDK tool
runners — is *this loop* with conveniences bolted on. Build it once, raw, and those
frameworks become legible instead of magical: you'll know exactly what they hide.

The loop has a small number of invariants, and **every production incident with
tool use traces back to violating one of them**:

- The assistant message carrying the `tool_use` blocks is appended to history
  **unmodified**.
- Every `tool_use_id` gets **exactly one** `tool_result`.
- All results for a turn go back in **one** `user` message.
- The loop is **bounded** — a `max_steps` circuit breaker so a confused model can't
  spin forever.

Get those four right and you have a foundation that holds under load.

## Objectives & prerequisites

**By the end you can:**

- Wrap a Python function as a `Tool` (schema + validated callable).
- Register tools and dispatch the model's requests by name, validating arguments
  with **Pydantic** before anything runs.
- Drive the full agent loop with a **max-steps** guard so it always terminates.
- Return tool failures as `is_error` *results* the model can recover from — never
  as exceptions that kill the loop.

**Prereqs:** 12-01 (the tool-use round-trip). Pydantic (in `requirements.txt`).

**Run first:** the Setup cell. Defaults to `MOCK=1`, so the whole multi-step run
is **free and offline**.

In [ ]:
# --- Setup -------------------------------------------------------------------
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv()  # never hardcode keys

# MOCK=1 (default): a scripted multi-step model drives the loop with canned,
# realistic tool-use blocks -> free, offline, deterministic. MOCK=0 hits the
# live Anthropic API (a short multi-step run, a handful of calls).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(12)  # determinism for anything stochastic here

MODEL = "claude-opus-4-8"  # the book's default model
DATA_DIR = Path("data")    # tiny committed fixtures live here

if MOCK:
    print("MOCK mode: scripted model. No API key needed, nothing billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env or use COMPANION_MOCK=1."
        )
    import anthropic

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
    print(f"LIVE mode: calling {MODEL}. A short multi-step run will be billed.")

## 1 · A `Tool` abstraction: schema + a validated callable

A tool is two things bound together: the **schema** the model sees, and the
**Python callable** you run when it's chosen — plus a **Pydantic model** that
validates the model's arguments *before* the callable ever executes. Schema
conformance from the provider is the floor; your validation is the ceiling (only
*your* code knows a path must stay inside the sandbox).

In [ ]:
class Tool:
    """Binds a model-facing schema to a validated Python callable.

    `args_model` is a Pydantic model: it validates (and coerces) the arguments the
    model sends before we call `fn`. The `schema` property is what we hand to the
    API.
    """

    def __init__(self, name, description, args_model, fn):
        self.name = name
        self.description = description
        self.args_model = args_model
        self.fn = fn

    @property
    def schema(self):
        # Pydantic emits JSON Schema; the Anthropic API wants it under input_schema.
        input_schema = self.args_model.model_json_schema()
        input_schema.pop("title", None)  # cosmetic: drop the model class name
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": input_schema,
        }

    def run(self, raw_args: dict) -> str:
        """Validate args, then execute. Raises ValidationError on bad input."""
        validated = self.args_model(**raw_args)   # <-- the security gate
        return self.fn(validated)

## 2 · Two real tools: a safe web-fetch stub and a file reader

Now two tools with teeth. The web-fetch is a **stub** (no real network in MOCK —
it returns canned content), so the notebook stays offline and deterministic. The
file reader reads the small fixtures in `data/`, and it **validates the path stays
inside `data/`** — a model-supplied path is untrusted input, and path traversal is
the classic way a file tool becomes a data leak.

In [ ]:
# Canned "web" content so the stub is offline + deterministic.
_FAKE_WEB = {
    "https://example.com/status": "All systems operational. Last incident: none.",
    "https://example.com/pricing": "Pro plan: $20/mo. Team plan: $40/user/mo.",
}


class FetchArgs(BaseModel):
    url: str = Field(description="Absolute http(s) URL to fetch.")


def _fetch(args: FetchArgs) -> str:
    """Safe web-fetch STUB. Returns canned content; no real network call.

    A real version (see the blueprint) would use httpx with a timeout, an
    allowlist, and a response-size cap.
    """
    if not args.url.startswith(("http://", "https://")):
        raise ValueError("url must start with http:// or https://")
    return _FAKE_WEB.get(args.url, f"[stub] no canned content for {args.url}")


class ReadFileArgs(BaseModel):
    name: str = Field(description="File name inside the data/ folder, e.g. 'refund-policy.md'.")


def _read_file(args: ReadFileArgs) -> str:
    """Read a fixture from data/, refusing any path that escapes the folder."""
    target = (DATA_DIR / args.name).resolve()
    root = DATA_DIR.resolve()
    if root not in target.parents and target != root:
        # Path traversal attempt, e.g. name="../../secrets". Refuse loudly.
        raise ValueError(f"path escapes the data/ sandbox: {args.name!r}")
    if not target.exists():
        raise FileNotFoundError(f"no such fixture: {args.name!r}")
    return target.read_text(encoding="utf-8")


fetch_tool = Tool(
    name="fetch_url",
    description=(
        "Fetch the text content at an http(s) URL. Call this when the user asks "
        "about live web content you do not already know."
    ),
    args_model=FetchArgs,
    fn=_fetch,
)

read_file_tool = Tool(
    name="read_file",
    description=(
        "Read a policy document from the local docs folder. Call this whenever the "
        "user asks about refunds, cancellation, or other documented policy. Valid "
        "names include 'refund-policy.md' and 'cancellation-policy.md'."
    ),
    args_model=ReadFileArgs,
    fn=_read_file,
)

print("fetch_url schema:")
print(json.dumps(fetch_tool.schema, indent=2))

## 3 · A registry and a dispatcher that never raises

The **registry** maps a tool name to its `Tool`. The **dispatcher** (`run_tool`)
is the one chokepoint where execution happens — and it **never raises**. A failure
becomes a `(message, is_error=True)` pair, because the API rejects your next
request if any `tool_use_id` lacks a result. "Never raises" is what guarantees
every request gets answered, even when the tool blows up.

In [ ]:
REGISTRY = {t.name: t for t in (fetch_tool, read_file_tool)}
TOOLS = [t.schema for t in REGISTRY.values()]  # what we send the model each turn


def run_tool(name: str, raw_args: dict) -> tuple[str, bool]:
    """Execute one tool. Return (result_text, is_error). NEVER raises.

    Validation errors and runtime errors both come back as is_error=True with a
    message phrased FOR THE MODEL, so it can correct itself on the next turn.
    """
    tool = REGISTRY.get(name)
    if tool is None:
        return f"Unknown tool: {name!r}. Available: {list(REGISTRY)}", True
    try:
        return tool.run(raw_args), False
    except ValidationError as exc:
        # Pydantic's message tells the model exactly which field was wrong.
        return f"Invalid arguments for {name}: {exc.errors()}", True
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}", True


# Quick smoke test of the dispatcher (no model involved yet).
print(run_tool("read_file", {"name": "refund-policy.md"})[0][:80], "...")
print("error path:", run_tool("read_file", {"name": "does-not-exist.md"}))

## 4 · The agent loop

Here is the whole engine. Call the model; if it stopped to use tools, execute
every requested tool, append **all** results in one user message, and loop; if it
stopped with `end_turn`, return the text. The `for _ in range(max_steps)` is the
circuit breaker — the loop *always* terminates, even against a model that would
otherwise ping-pong forever.

In [ ]:
SYSTEM = (
    "You are the support agent for our platform. Ground every answer in the "
    "policy docs via read_file; do not invent policy."
)


def _call_model(messages):
    """One model turn. MOCK returns scripted dicts; live calls the SDK and
    normalizes the response into the same {stop_reason, content} dict shape."""
    if MOCK:
        return _scripted_turn(messages)
    resp = client.messages.create(
        model=MODEL, max_tokens=2048, system=SYSTEM, tools=TOOLS, messages=messages,
    )
    content = []
    for b in resp.content:
        if b.type == "text":
            content.append({"type": "text", "text": b.text})
        elif b.type == "tool_use":
            content.append({"type": "tool_use", "id": b.id, "name": b.name, "input": b.input})
    return {"stop_reason": resp.stop_reason, "content": content}


def agent(user_input: str, max_steps: int = 10) -> str:
    messages = [{"role": "user", "content": user_input}]
    for step in range(max_steps):              # hard bound: no infinite loops
        response = _call_model(messages)
        if response["stop_reason"] != "tool_use":
            # Model is done: return its final text (or empty if it produced none).
            return next(
                (b["text"] for b in response["content"] if b["type"] == "text"),
                "",
            )
        # Echo the assistant turn (with its tool_use blocks) VERBATIM.
        messages.append({"role": "assistant", "content": response["content"]})
        results = []
        for block in response["content"]:
            if block["type"] != "tool_use":
                continue
            output, is_error = run_tool(block["name"], block["input"])
            results.append({
                "type": "tool_result",
                "tool_use_id": block["id"],     # one result per tool_use_id
                "content": output,
                "is_error": is_error,
            })
        # All results for the turn go back in ONE user message.
        messages.append({"role": "user", "content": results})
    return "Stopped: step limit reached without a final answer."

### The scripted model (MOCK)

So the loop runs offline, here's a tiny scripted "model": it looks at the last
message and returns the next canned turn. It deliberately makes a **bad call
first** (a path-traversal `name`), sees the validation error, then **self-corrects**
— exactly the recovery pattern real models exhibit.

In [ ]:
def _last_tool_result_texts(messages):
    """Pull the text of tool_results in the most recent user message, if any."""
    if not messages:
        return []
    last = messages[-1]
    if last["role"] != "user" or not isinstance(last["content"], list):
        return []
    return [c.get("content", "") for c in last["content"] if c.get("type") == "tool_result"]


def _scripted_turn(messages):
    """A deterministic stand-in for the model that drives a 3-step run:

    turn 1: ask to read a policy file -- but with a BAD (traversal) path.
    turn 2: after the validation error, retry with the CORRECT name.
    turn 3: after the file content, answer in prose (end_turn).
    """
    results = _last_tool_result_texts(messages)

    # turn 3: we've successfully read the file -> final answer.
    if results and any("full refund" in r for r in results):
        return {
            "stop_reason": "end_turn",
            "content": [{
                "type": "text",
                "text": ("Yes -- you can get a full refund within 30 days of "
                         "purchase, issued to your original payment method."),
            }],
        }

    # turn 2: the previous call errored (bad path) -> retry with the right name.
    if results and any("escapes the data/ sandbox" in r for r in results):
        return {
            "stop_reason": "tool_use",
            "content": [
                {"type": "text", "text": "Let me read the correct policy file."},
                {"type": "tool_use", "id": "toolu_read_ok",
                 "name": "read_file", "input": {"name": "refund-policy.md"}},
            ],
        }

    # turn 1: first attempt -- a plausible but UNSAFE path (predict the failure!).
    return {
        "stop_reason": "tool_use",
        "content": [
            {"type": "text", "text": "I'll check the refund policy."},
            {"type": "tool_use", "id": "toolu_read_bad",
             "name": "read_file", "input": {"name": "../refund-policy.md"}},
        ],
    }

## 5 · 🔮 Predict: the model's first call uses a bad path

Run the agent on *"Can I get a refund?"*. On turn 1 the (scripted) model asks to
`read_file` with `name="../refund-policy.md"` — a path that tries to climb out of
`data/`.

**Predict before running:** What does `run_tool` return for that call — does it
crash the loop, or come back as an error result? And what does the model do on the
*next* turn once it sees that error?

In [ ]:
answer = agent("Can I get a refund?")
print("FINAL ANSWER:\n", answer)

**What you just saw.** The bad path didn't crash anything. `run_tool` caught the
`ValueError` from the sandbox check and returned it as an `is_error` result with a
message the model could read. The model then retried with the correct file name,
got the policy text, and answered. **Error recovery is a dialogue** — your job is
to keep it alive with informative results, not end it with exceptions.

## 6 · ⚠️ Pitfall: forgetting the termination guard

The single scariest bug in a hand-rolled loop is the one that never ends: two
unhelpful tools, a model that keeps trying, and a bill that climbs while you sleep.
The fix is the `max_steps` bound you already have. Prove it works by capping steps
hard and feeding a model that *never* says it's done.

In [ ]:
def _never_done_turn(_messages):
    """A pathological model that asks for a tool forever."""
    return {
        "stop_reason": "tool_use",
        "content": [{"type": "tool_use", "id": "toolu_loop",
                     "name": "read_file", "input": {"name": "refund-policy.md"}}],
    }


# Temporarily swap in the pathological model and cap the loop at 3 steps.
_real_scripted = _scripted_turn
try:
    globals()["_scripted_turn"] = _never_done_turn
    result = agent("loop forever please", max_steps=3)
finally:
    globals()["_scripted_turn"] = _real_scripted  # restore

print(result)
# Without max_steps this would never return. WITH it, the loop is a guaranteed
# circuit breaker: bounded cost, bounded latency, every time.

## 🎯 Senior lens: idempotency and side effects

The deepest design split in your toolbox is **reads vs writes**. Reads
(`read_file`, `fetch_url`, `search_docs`) are safe to call speculatively, in
parallel, and on retry — there's no harm in doing one twice. Writes
(`create_ticket`, `send_email`, `issue_refund`) are not. Agent loops *retry* — your
network layer retries, the model retries when a result looks wrong, and users
double-click — so any tool with side effects must be **idempotent**: it should
accept or derive an idempotency key so "create the ticket" run twice yields one
ticket. Name tools honestly, too: an `update_record` that also emails the customer
will eventually surprise everyone, the model included. Review your tool surface the
way you review an IAM policy — assume the worst call *will* happen.

## Recap

- A **`Tool`** binds a model-facing schema to a validated callable; **Pydantic**
  validates arguments before execution — the real security gate beyond the schema.
- The **dispatcher never raises**: failures become `is_error` results, so every
  `tool_use_id` gets exactly one `tool_result` and the loop survives.
- The **loop invariants**: echo the assistant turn verbatim, one result per id, all
  results in one user message, and a **`max_steps` bound** that guarantees
  termination.
- **Recovery is a dialogue** — a clear error message gets a corrected retry; a raw
  exception gets a dead loop.
- **Reads are safe to retry; writes need idempotency keys** and often a human gate.

## Exercises

Use the empty cells below. (Solutions land in `solutions/` in Phase 2.)

1. **Add a write tool.** Implement `create_ticket(title, body, idempotency_key)`
   that returns a stable `ticket_id` derived from the key (e.g.
   `f"TCK-{abs(hash(key)) % 10000}"`). Predict: call it twice with the same key —
   do you get one ticket id or two? Why does that property matter in a retrying
   loop?
2. **Tighten validation.** Add a `max_results: int = Field(ge=1, le=20)` field to a
   tool and have the scripted model send `max_results=999`. Predict the exact
   `is_error` message the model will see, then confirm.
3. **Lower the guard.** Set `max_steps=1` on the refund question. Predict whether
   the agent reaches a final answer, and explain what the returned string tells you
   about why bounding the loop is non-negotiable.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

## Next

You built the toy. Now meet the real one, and learn what breaks when tools run in
parallel.

- ▶️ **Next notebook:** [`12-03-parallel-tools-and-recovery.ipynb`](./12-03-parallel-tools-and-recovery.ipynb)
  — multiple tool calls in one turn, concurrency with `asyncio`, and surviving
  partial failure.
- 🔧 **Blueprint (the production version of *this* loop):**
  [`../../blueprints/agent-loop/`](../../blueprints/agent-loop/) — typed
  tools, retries, telemetry hooks. You built the toy; that's the real one.
- 🧩 **Template:** the tool-definition pattern here feeds
  [`../../templates/agent-project-starter/`](../../templates/agent-project-starter/).
- 🎓 **Capstone:** this is `capstone-project/agents/raw/loop.py` and
  `capstone-project/agents/tools/`; checkpoint `checkpoints/ch12-tool-loop`.
- 📖 **Book:** see §12.3 (safe execution) and §12.4 (the Build).